# Memory Impact Evaluation

**Objective:** Quantify improvement in agent decisions with memory vs memoryless baseline.

**Ticket:** REC-321

**Spec:** `reports/2026-02-19/memory_evaluation_spec.md`

## 1. Setup & Imports

In [ ]:
# Standard imports
import os
import sys
import json
import yaml
import random
import warnings
from pathlib import Path
from datetime import datetime, timedelta
from typing import List, Dict, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set up paths
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
BACKEND_SRC = PROJECT_ROOT / 'backend' / 'src'

# Add backend to path for imports
sys.path.insert(0, str(BACKEND_SRC))
sys.path.insert(0, str(PROJECT_ROOT / 'backend'))

# Suppress warnings
warnings.filterwarnings('ignore')

# Set random seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"Project root: {PROJECT_ROOT}")
print(f"Backend src: {BACKEND_SRC}")

In [ ]:
# Load config
with open('config.yaml', 'r') as f:
    CONFIG = yaml.safe_load(f)

print("Config loaded:")
print(json.dumps(CONFIG, indent=2, default=str))

In [ ]:
# Import backend modules
try:
    from agent.memory import AgentMemory, Decision, Memory
    print("✓ agent.memory imported")
except ImportError as e:
    print(f"✗ agent.memory: {e}")

try:
    from agent.context import ContextAggregator
    print("✓ agent.context imported")
except ImportError as e:
    print(f"✗ agent.context: {e}")

try:
    from backtest.historical_scores import HistoricalScoreGenerator
    print("✓ backtest.historical_scores imported")
except ImportError as e:
    print(f"✗ backtest.historical_scores: {e}")

try:
    from backtest.engine import BacktestEngine
    print("✓ backtest.engine imported")
except ImportError as e:
    print(f"✗ backtest.engine: {e}")

try:
    from backtest.metrics import calculate_metrics
    print("✓ backtest.metrics imported")
except ImportError as e:
    print(f"✗ backtest.metrics: {e}")

In [ ]:
# Local embedding model (for cost-free experimentation)
from sentence_transformers import SentenceTransformer

EMBED_MODEL = SentenceTransformer(CONFIG['memory']['embedding_model'])
print(f"✓ Loaded embedding model: {CONFIG['memory']['embedding_model']}")
print(f"  Embedding dimension: {EMBED_MODEL.get_sentence_embedding_dimension()}")

## 2. Data Loading

In [ ]:
# TODO: Load Kaggle sentiment dataset
# Path: backend/data/kaggle_news/ or download

# For now, check what data we have
DATA_DIR = PROJECT_ROOT / 'backend' / 'data'
print("Available data files:")
for f in DATA_DIR.glob('*.db'):
    print(f"  - {f.name}")
for f in DATA_DIR.glob('*.json'):
    print(f"  - {f.name}")

In [ ]:
# Load historical scores if available
# TODO: Generate historical scores using HistoricalScoreGenerator

TRAINING_START = pd.to_datetime(CONFIG['data']['training_start'])
TRAINING_END = pd.to_datetime(CONFIG['data']['training_end'])
TEST_START = pd.to_datetime(CONFIG['data']['test_start'])
TEST_END = pd.to_datetime(CONFIG['data']['test_end'])

print(f"Training period: {TRAINING_START.date()} to {TRAINING_END.date()}")
print(f"Test period: {TEST_START.date()} to {TEST_END.date()}")

## 3. Memory Builder

Build the memory database from training period decisions.

In [ ]:
def decision_to_text(d: Dict) -> str:
    """Convert decision dict to text for embedding."""
    return f"""
    Ticker: {d['ticker']}
    Sector: {d.get('sector', 'Unknown')}
    Action: {d['action']}
    Score: {d['score']}
    Regime: {d.get('regime', 'normal')}
    Rationale: {d.get('rationale', '')}
    Outcome: {d.get('outcome_pct', 0):+.1f}%
    Lesson: {d.get('lesson_learned', '')}
    """.strip()

def embed(text: str) -> np.ndarray:
    """Generate embedding for text."""
    return EMBED_MODEL.encode(text)

# Test embedding
test_text = decision_to_text({
    'ticker': 'AAPL',
    'action': 'BUY',
    'score': 85,
    'sector': 'Technology',
    'regime': 'normal',
    'outcome_pct': 12.5,
    'lesson_learned': 'Tech with high scores in normal regime performs well'
})
test_embedding = embed(test_text)
print(f"Test embedding shape: {test_embedding.shape}")

In [ ]:
# TODO: Build memory from training period
# 1. Generate weekly scores for training period
# 2. For each week, identify BUY candidates (score >= 70)
# 3. Simulate decision and record outcome 4 weeks later
# 4. Generate lesson learned
# 5. Embed and store

print("Memory builder: TODO - implement after data loading")

## 4. Baseline Agent (No Memory)

In [ ]:
SYSTEM_PROMPT_NO_MEMORY = """
You are Sigil's autonomous trading agent. You make weekly BUY/SELL decisions.

CONTEXT (Current):
{context}

Based on the current context, decide:
1. Which stocks to BUY (if any)
2. Which stocks to SELL (if any)
3. Your confidence (0-1) and rationale

Respond in JSON format:
{{
  "buys": [{{"ticker": "XXX", "confidence": 0.8, "rationale": "..."}}],
  "sells": [{{"ticker": "YYY", "confidence": 0.7, "rationale": "..."}}]
}}
"""

print("Baseline prompt defined")

In [ ]:
# TODO: Run baseline agent on test period
# For each week:
#   1. Build context from scores
#   2. Call Claude with SYSTEM_PROMPT_NO_MEMORY
#   3. Parse decisions
#   4. Simulate and record

baseline_results = []
print("Baseline agent: TODO - implement after memory builder")

## 5. Memory Agent

In [ ]:
SYSTEM_PROMPT_WITH_MEMORY = """
You are Sigil's autonomous trading agent. You make weekly BUY/SELL decisions.

CONTEXT (Current):
{context}

SIMILAR PAST DECISIONS:
{memories}

Based on your past experience and current context, decide:
1. Which stocks to BUY (if any)
2. Which stocks to SELL (if any)
3. Your confidence (0-1) and rationale

Consider what worked and didn't work in similar past situations.

Respond in JSON format:
{{
  "buys": [{{"ticker": "XXX", "confidence": 0.8, "rationale": "..."}}],
  "sells": [{{"ticker": "YYY", "confidence": 0.7, "rationale": "..."}}],
  "memories_used": ["reference to relevant past decision..."]
}}
"""

print("Memory-enhanced prompt defined")

In [ ]:
def retrieve_memories(query_embedding: np.ndarray, memory_db: List[Dict], top_k: int = 5) -> List[Dict]:
    """Retrieve top-k similar memories."""
    similarities = []
    for mem in memory_db:
        sim = np.dot(query_embedding, mem['embedding']) / (
            np.linalg.norm(query_embedding) * np.linalg.norm(mem['embedding'])
        )
        similarities.append((sim, mem))
    
    # Sort by similarity descending
    similarities.sort(key=lambda x: x[0], reverse=True)
    
    return [{'similarity': sim, **mem} for sim, mem in similarities[:top_k]]

print("Memory retrieval function defined")

In [ ]:
# TODO: Run memory agent on test period
# For each week:
#   1. Build context from scores
#   2. Retrieve similar memories
#   3. Call Claude with SYSTEM_PROMPT_WITH_MEMORY
#   4. Parse decisions
#   5. Simulate and record

memory_results = []
print("Memory agent: TODO - implement after baseline")

## 6. Analysis

In [ ]:
def calculate_comparison_metrics(baseline: List[Dict], memory: List[Dict]) -> pd.DataFrame:
    """Calculate and compare metrics between baseline and memory agents."""
    
    def calc_metrics(results: List[Dict]) -> Dict:
        if not results:
            return {}
        
        returns = [r['outcome_pct'] for r in results if r.get('outcome_pct') is not None]
        wins = [r for r in returns if r > 0]
        
        return {
            'total_trades': len(results),
            'win_rate': len(wins) / len(returns) * 100 if returns else 0,
            'avg_return': np.mean(returns) if returns else 0,
            'sharpe': np.mean(returns) / np.std(returns) * np.sqrt(52) if returns and np.std(returns) > 0 else 0,
            'max_drawdown': min(returns) if returns else 0,
            'total_return': sum(returns) if returns else 0,
        }
    
    baseline_metrics = calc_metrics(baseline)
    memory_metrics = calc_metrics(memory)
    
    df = pd.DataFrame({
        'Metric': list(baseline_metrics.keys()),
        'Baseline': list(baseline_metrics.values()),
        'Memory': list(memory_metrics.values()),
    })
    df['Improvement'] = df['Memory'] - df['Baseline']
    
    return df

print("Comparison metrics function defined")

In [ ]:
# TODO: Run analysis after both agents complete
# comparison_df = calculate_comparison_metrics(baseline_results, memory_results)
# display(comparison_df)

print("Analysis: TODO - run after agents complete")

## 7. Conclusions

TODO: Fill in after running experiments

In [ ]:
# Save results
# results = {
#     'baseline': baseline_results,
#     'memory': memory_results,
#     'comparison': comparison_df.to_dict(),
# }
# with open('results/experiment_results.json', 'w') as f:
#     json.dump(results, f, indent=2, default=str)

print("Notebook complete - TODO items remain for full implementation")